# Generic Sudoku as a CSP

An $n^2\times n^2$ Sudoku is a canonical example of a *constraint satisfaction problem* (CSP): the rule is always the same, *a set of cells must hold pairwise different values*, and what changes from row to row, column to column and block to block is only **which cells belong to the set**.

In this solution we build a single abstraction (`box`: a group of cells, some of which may be pinned to a value) and obtain rows, columns, blocks, random clues and several Sudoku variants simply as **different groups**, without ever touching the CSP model.

We use the [OR-Tools](https://developers.google.com/optimization) library and its **CP-SAT** solver.

## Problem analysis

### Choice of solving technique

Chapter 2 distinguishes integer programming problems (linear constraints over integers, like those of the previous worksheet, solved with SCIP) from *Constraint Programming* (CP) problems, whose constraints go beyond the descriptive power of LIA. The constraint $\mathrm{alldifferent}(S)$ belongs to this second class.

Given a group $S$ of cells, $\mathrm{alldifferent}(S)$ is satisfied by an assignment $\sigma$ when
$$\mathrm{alldifferent}(S) \;\equiv\; \bigwedge_{x\in S}\ \bigwedge_{y<x} \sigma(x)\neq\sigma(y)$$
In LIA, each $x\neq y$ is a disjunction ($x<y \vee x>y$), so it has no direct representation in an integer programming model; it would have to be encoded with auxiliary binary variables.

**CP-SAT** supports this constraint directly (`AddAllDifferent`) and, as noted in chapter 2, uses strategies specific to it to prune the search tree (propagation), which do not apply to generic constraints. We therefore chose CP-SAT, which is also the course's suggestion.

An alternative we do not follow: a binary integer programming model with variables $X_{i,j,v}=1$ iff cell $(i,j)$ holds value $v$. This model is a *set partition* problem ($A\cdot Y=1$) with $N^3$ variables and can be solved with SCIP. It is linear, but it loses the direct correspondence between "group of cells" and "constraint".

### Model

Let $N=n^2$. We use a family of integer variables
$$x_{i,j}\in[1,N]\qquad i,j\in[0..N\!-\!1]$$
where $x_{i,j}$ is the value of cell $(i,j)$. A **group** $g$ is a set of cells $C_g$ together with a partial function $v_g:C_g\rightharpoonup[1,N]$ that pins some of them. Each group imposes:

$$\mathrm{alldifferent}\big(\{x_{i,j}\mid (i,j)\in C_g\}\big) \qquad\wedge\qquad \forall (i,j)\in\mathrm{dom}(v_g)\cdot\ x_{i,j}=v_g(i,j)$$

Classic Sudoku is then the union of the groups: the $N$ rows, the $N$ columns, the $N$ blocks of size $n\times n$, and a group of clues. The clues only pin values: we do not impose *alldifferent* on them (see the next note).

### Note on the clue group

The random clues are represented by a `box` (as the assignment requires). But the clues do **not** have to be pairwise different (two clues may legitimately both be 5 in different rows and columns). So the model distinguishes two roles for a group:

- `add_groups(...)`: imposes *alldifferent* on the group's cells **and** pins the ones that have a value;
- `add_clues(...)`: only pins the cells that have a value.

This way the model still does not know where a group came from: it is only told *how* to use it.

## Implementation

We start by importing the libraries.

In [1]:
import random
import time
from ortools.sat.python import cp_model

### Exercise 1: `box`, the generic cell group (R1)

The class stores a mapping $(i,j)\mapsto$ value or `None` (`None` = free cell). It knows nothing about rows, columns or blocks. The parameter `n` defines the $n^2\times n^2$ grid and is used only to validate coordinates and values.

The method `add(i, j, val=None)` rejects coordinates outside $[0,N[$ (`IndexError`) and values outside $[1,N]$ (`ValueError`). Adding a cell that already exists without a value does not erase the value it had; two different values for the same cell raise `ValueError`.

In [2]:
class box:
    # Group of cells (i,j) -> value | None, subject to 'all different'.

    def __init__(self, n, cells=None):
        if not isinstance(n, int) or n < 1:
            raise ValueError(f"invalid n: {n!r}")
        self.n = n
        self.N = n * n
        self.cells = {}
        for (i, j), val in (cells or {}).items():
            self.add(i, j, val)

    def add(self, i, j, val=None):
        if not (isinstance(i, int) and isinstance(j, int)):
            raise TypeError("coordinates must be integers")
        if not (0 <= i < self.N and 0 <= j < self.N):
            raise IndexError(f"cell ({i},{j}) is outside the {self.N}x{self.N} grid")
        if val is not None:
            if not isinstance(val, int) or not (1 <= val <= self.N):
                raise ValueError(f"value {val!r} is outside [1,{self.N}]")
            old = self.cells.get((i, j))
            if old is not None and old != val:
                raise ValueError(f"cell ({i},{j}) is already pinned to {old}")
            self.cells[(i, j)] = val
        else:
            self.cells.setdefault((i, j), None)

    def matrix(self):
        # N x N matrix: pinned value in pinned cells, 0 everywhere else.
        m = [[0] * self.N for _ in range(self.N)]
        for (i, j), val in self.cells.items():
            if val is not None:
                m[i][j] = val
        return m

    def pinned(self):
        return {c: v for c, v in self.cells.items() if v is not None}

    def __len__(self):
        return len(self.cells)

    def __repr__(self):
        return f"{type(self).__name__}(n={self.n}, cells={self.cells})"

### Exercise 2: `cube` and `path` (R2, R3)

**`cube(n, i, j)`** is the $n\times n$ block whose top-left corner is $(i\cdot n,\ j\cdot n)$, with $0\le i,j<n$:
$$C_{\mathrm{cube}(i,j)}=\{(i\cdot n+a,\ j\cdot n+b)\mid a,b<n\}$$

**`path(n, start, end)`** is the straight segment, inclusive at both ends, between two cells of the same row or the same column. The direction of travel is given by the sign of the differences, so it works with `end` after or before `start`:
$$d_i=\mathrm{sgn}(i_1-i_0),\quad d_j=\mathrm{sgn}(j_1-j_0),\quad C_{\mathrm{path}}=\{(i_0+t\,d_i,\ j_0+t\,d_j)\mid t\le\max(|i_1-i_0|,|j_1-j_0|)\}$$

A segment that is neither horizontal nor vertical is rejected.

In [3]:
class cube(box):
    # n x n block with block indices (i,j), top-left corner at (i*n, j*n).

    def __init__(self, n, i, j):
        if not (0 <= i < n and 0 <= j < n):
            raise IndexError(f"block ({i},{j}) is outside [0,{n}[")
        super().__init__(n)
        self.block = (i, j)
        for a in range(n):
            for b in range(n):
                self.add(i * n + a, j * n + b)


class path(box):
    # Straight segment (horizontal or vertical) from start to end, inclusive.

    def __init__(self, n, start, end):
        super().__init__(n)
        (i0, j0), (i1, j1) = start, end
        if i0 != i1 and j0 != j1:
            raise ValueError(f"{start} -> {end} is neither horizontal nor vertical")
        di = (i1 > i0) - (i1 < i0)
        dj = (j1 > j0) - (j1 < j0)
        for t in range(max(abs(i1 - i0), abs(j1 - j0)) + 1):
            self.add(i0 + t * di, j0 + t * dj)

In [4]:
# Examples
print(sorted(cube(3, 1, 2).cells))            # middle-right block
print(sorted(path(3, (0, 0), (0, 8)).cells))  # row 0, left to right
print(sorted(path(3, (8, 4), (0, 4)).cells))  # column 4, bottom to top

[(3, 6), (3, 7), (3, 8), (4, 6), (4, 7), (4, 8), (5, 6), (5, 7), (5, 8)]
[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8)]
[(0, 4), (1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4), (7, 4), (8, 4)]


### Exercise 3: Random clue generation (R4)

The function returns a `box` with $k$ distinct cells chosen at random, each pinned to a random value in $[1,N]$. By default $k=n$, as the assignment suggests. No new class is needed.

The random generator is injectable (`rng`) so that tests are reproducible.

In [5]:
def random_clues(n, k=None, rng=None):
    rng = rng or random
    N = n * n
    if k is None:
        k = n
    if not (0 <= k <= N * N):
        raise ValueError(f"k={k} is outside [0,{N*N}]")
    g = box(n)
    all_cells = [(i, j) for i in range(N) for j in range(N)]
    for i, j in rng.sample(all_cells, k):
        g.add(i, j, rng.randint(1, N))
    return g

c = random_clues(3, rng=random.Random(1))
print(c.pinned())

{(1, 8): 5, (8, 0): 2, (0, 8): 8}


### Exercise 4: CSP model (R5)

The model creates one integer variable $x_{i,j}\in[1,N]$ per cell. It has:

- `add_groups(*gs)`: for each group it imposes `AddAllDifferent` on the variables of its cells and pins those that have a value. It accepts an arbitrary number of groups and does not distinguish their origin (`box`, `cube`, `path`, ...);
- `add_clues(*gs)`: only pins the cells that have a value (see the note on clues);
- `solve()`: returns the grid as a list of lists, or `None` if the puzzle has no solution. The attribute `status` stores the solver status (`'OPTIMAL'`, `'INFEASIBLE'`, `'UNKNOWN'`, ...), which makes it possible to tell *no solution* apart from *time limit reached*.

In [6]:
class sudoku_model:

    def __init__(self, n):
        self.n = n
        self.N = n * n
        self.m = cp_model.CpModel()
        self.x = [[self.m.NewIntVar(1, self.N, f"x_{i}_{j}")
                   for j in range(self.N)] for i in range(self.N)]
        self.status = None

    def _check(self, g):
        if g.n != self.n:
            raise ValueError(f"group with n={g.n} in a model with n={self.n}")

    def _pin(self, g):
        for (i, j), val in g.cells.items():
            if val is not None:
                self.m.Add(self.x[i][j] == val)

    def add_groups(self, *groups):
        for g in groups:
            self._check(g)
            if len(g) > 1:
                self.m.AddAllDifferent([self.x[i][j] for (i, j) in g.cells])
            self._pin(g)

    def add_clues(self, *groups):
        for g in groups:
            self._check(g)
            self._pin(g)

    def solve(self, time_limit=None):
        solver = cp_model.CpSolver()
        if time_limit is not None:
            solver.parameters.max_time_in_seconds = time_limit
        st = solver.Solve(self.m)
        self.status = solver.StatusName(st)
        if st in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            return [[solver.Value(self.x[i][j]) for j in range(self.N)]
                    for i in range(self.N)]
        return None

### Exercise 5: Full Sudoku (R6)

An $n^2\times n^2$ Sudoku is the union of all rows, all columns, all blocks and (at least) one group of random clues. Rows use `path` from left to right and columns from bottom to top, to exercise both directions.

`full_sudoku` also accepts extra groups (`extra`), which the optional variants will use.

Since the clues are chosen at random, they may be contradictory (for example two 5s in the same row) and the puzzle may have no solution. **Chosen strategy:** `generate_and_solve` tries new clues until it gets a solvable puzzle (at most `attempts`). Justification: with small $k$ most puzzles are solvable (we measure this in exercise 7), each attempt is cheap, and the goal is to obtain a filled grid to validate. If the attempts run out, it returns `None`.

In [7]:
def full_sudoku(n, clues, extra=()):
    N = n * n
    rows   = [path(n, (i, 0), (i, N - 1)) for i in range(N)]
    cols   = [path(n, (N - 1, j), (0, j)) for j in range(N)]
    blocks = [cube(n, i, j) for i in range(n) for j in range(n)]
    m = sudoku_model(n)
    m.add_groups(*rows, *cols, *blocks, *extra)
    m.add_clues(clues)
    return m


def generate_and_solve(n, k=None, attempts=100, seed=None, extra=(), time_limit=None):
    # Returns (clues, grid, n_attempts) or (None, None, attempts).
    rng = random.Random(seed)
    for t in range(1, attempts + 1):
        clues = random_clues(n, k, rng)
        grid = full_sudoku(n, clues, extra).solve(time_limit)
        if grid is not None:
            return clues, grid, t
    return None, None, attempts

A helper function to display the grid as text, with separators between blocks. Clues are marked with `*`.

In [8]:
def show(grid, n, clues=None):
    N = n * n
    pinned = clues.pinned() if clues else {}
    w = len(str(N))
    sep = "+" + "+".join("-" * ((w + 2) * n) for _ in range(n)) + "+"
    for i in range(N):
        if i % n == 0:
            print(sep)
        line = "|"
        for j in range(N):
            mark = "*" if (i, j) in pinned else " "
            line += f"{grid[i][j]:>{w}}{mark}"
            if (j + 1) % n == 0:
                line += "|"
            else:
                line += " "
        print(line)
    print(sep)

In [9]:
clues, grid, t = generate_and_solve(3, seed=42)
print("attempts:", t, "| clues:", clues.pinned())
show(grid, 3, clues)

attempts: 2 | clues: {(1, 4): 7, (7, 6): 1, (1, 2): 1}
+---------+---------+---------+
|5  2  9 |3  6  4 |7  1  8 |
|3  4  1*|2  7* 8 |5  9  6 |
|7  6  8 |5  1  9 |2  4  3 |
+---------+---------+---------+
|2  7  3 |1  9  6 |8  5  4 |
|1  5  4 |8  2  3 |6  7  9 |
|8  9  6 |7  4  5 |3  2  1 |
+---------+---------+---------+
|6  1  5 |4  3  2 |9  8  7 |
|4  3  2 |9  8  7 |1* 6  5 |
|9  8  7 |6  5  1 |4  3  2 |
+---------+---------+---------+


### Exercise 6: Automatic validation

For a solved grid we check that

- every row, column and block contains exactly $\{1..N\}$, with no repetitions;
- the cells pinned by the clues keep their value;

and we test that `add` rejects coordinates outside the grid and values outside $[1,N]$, that `path` works in both directions, and that the model reports an impossible puzzle in a distinguishable way.

In [10]:
def validate(grid, n, clues=None):
    N = n * n
    target = set(range(1, N + 1))
    assert len(grid) == N and all(len(row) == N for row in grid)
    for i in range(N):
        assert set(grid[i]) == target, f"row {i}"
    for j in range(N):
        assert {grid[i][j] for i in range(N)} == target, f"column {j}"
    for bi in range(n):
        for bj in range(n):
            blk = {grid[bi*n+a][bj*n+b] for a in range(n) for b in range(n)}
            assert blk == target, f"block ({bi},{bj})"
    if clues:
        for (i, j), v in clues.pinned().items():
            assert grid[i][j] == v, f"clue ({i},{j})={v} was changed"
    return True


def must_raise(exc, f, *args):
    try:
        f(*args)
    except exc:
        return True
    raise AssertionError(f"{f} {args} should raise {exc.__name__}")


def run_tests():
    # --- box.add rejects invalid input
    b = box(3)
    must_raise(IndexError, b.add, 9, 0)
    must_raise(IndexError, b.add, 0, -1)
    must_raise(ValueError, b.add, 0, 0, 0)
    must_raise(ValueError, b.add, 0, 0, 10)
    b.add(0, 0, 5); b.add(0, 0)                  # None does not erase the 5
    assert b.cells[(0, 0)] == 5
    must_raise(ValueError, b.add, 0, 0, 6)       # conflicting value
    assert b.matrix()[0][0] == 5 and sum(map(sum, b.matrix())) == 5

    # --- cube and path
    assert sorted(cube(3, 1, 2).cells) == [(i, j) for i in range(3, 6) for j in range(6, 9)]
    assert sorted(path(3, (0, 0), (0, 8)).cells) == sorted(path(3, (0, 8), (0, 0)).cells)
    assert len(path(3, (8, 4), (0, 4))) == 9
    assert len(path(3, (2, 2), (2, 2))) == 1
    must_raise(ValueError, path, 3, (0, 0), (1, 1))
    must_raise(IndexError, path, 3, (0, 0), (0, 9))
    must_raise(IndexError, cube, 3, 3, 0)

    # --- random clues
    p = random_clues(3, rng=random.Random(0))
    assert len(p) == 3 and all(1 <= v <= 9 for v in p.pinned().values())
    assert len(random_clues(2)) == 2             # default k = n

    # --- model: "no solution" is distinguishable
    m = full_sudoku(2, box(2, {(0, 0): 1, (0, 1): 1}))
    assert m.solve() is None and m.status == "INFEASIBLE"

    # --- full flow for several n
    for n in (2, 3):
        for seed in range(20):
            clues, grid, _ = generate_and_solve(n, seed=seed)
            assert grid is not None
            validate(grid, n, clues)
    return "all tests passed"

print(run_tests())

all tests passed


### Exercise 7: Behaviour for $n=2$, $n=3$ and impossible puzzles

We confirm that nothing is hard-wired to $9\times9$, and we measure the fraction of random puzzles with no solution (which justifies the strategy of re-drawing the clues).

In [11]:
clues, grid, t = generate_and_solve(2, seed=7)
print("n=2, attempts:", t)
show(grid, 2, clues)
validate(grid, 2, clues)

n=2, attempts: 1
+------+------+
|4  3 |1* 2 |
|1  2 |3  4 |
+------+------+
|2  1 |4* 3 |
|3  4 |2  1 |
+------+------+


True

In [12]:
def impossible_fraction(n, k, samples=200):
    unsolvable = 0
    for seed in range(samples):
        clues = random_clues(n, k, random.Random(seed))
        if full_sudoku(n, clues).solve() is None:
            unsolvable += 1
    return unsolvable / samples

for n, k in [(2, 2), (2, 4), (3, 3), (3, 9), (3, 20)]:
    print(f"n={n}, k={k}: {impossible_fraction(n, k):.0%} with no solution")

n=2, k=2: 10% with no solution


n=2, k=4: 69% with no solution


n=3, k=3: 8% with no solution


n=3, k=9: 67% with no solution


n=3, k=20: 98% with no solution


With $k=n$ the fraction of impossible puzzles is small, so re-drawing the clues almost always converges on the first or second attempt. As $k$ grows, each new clue may contradict the previous ones and the fraction rises quickly: there, re-drawing clues stops being a good strategy (it would be better to generate the clues from a complete solution).

## Optional extensions

In every extension the CSP model (`sudoku_model`) stays **exactly the same**: we only add or replace groups.

### Diagonal Sudoku (X-Sudoku)

Two new `box`es, one per main diagonal.

In [13]:
def diagonals(n):
    N = n * n
    main, anti = box(n), box(n)
    for i in range(N):
        main.add(i, i)
        anti.add(i, N - 1 - i)
    return [main, anti]

def diagonals_ok(g, n):
    N = n * n
    return (len({g[i][i] for i in range(N)}) == N and
            len({g[i][N - 1 - i] for i in range(N)}) == N)

clues, grid, t = generate_and_solve(3, seed=3, extra=diagonals(3))
show(grid, 3, clues)
assert validate(grid, 3, clues) and diagonals_ok(grid, 3)

+---------+---------+---------+
|7  4  9 |5  6  8 |3  2  1 |
|5  2  8 |1  3  7 |4  6  9 |
|3  6  1 |4  2  9 |7  5  8 |
+---------+---------+---------+
|1  7  5 |3* 9  4 |6  8  2 |
|2  3  6 |7  8  5 |1  9  4 |
|8  9  4 |2  1  6 |5  3  7 |
+---------+---------+---------+
|4  1  3 |8  5  2 |9  7  6 |
|6  5  2 |9  7  1 |8* 4  3 |
|9  8  7 |6* 4  3 |2  1  5 |
+---------+---------+---------+


### Hyper-Sudoku / Windoku

Four extra $3\times3$ blocks with top-left corners at $(1,1)$, $(1,5)$, $(5,1)$ and $(5,5)$, overlapping the existing ones and not aligned with the block grid. For $n=3$:

In [14]:
def windows(n=3):
    N = n * n
    corners = [(1, 1), (1, N - n - 1), (N - n - 1, 1), (N - n - 1, N - n - 1)]
    gs = []
    for ci, cj in corners:
        g = box(n)
        for a in range(n):
            for b in range(n):
                g.add(ci + a, cj + b)
        gs.append(g)
    return gs

clues, grid, t = generate_and_solve(3, seed=5, extra=windows(3))
show(grid, 3, clues)
assert validate(grid, 3, clues)
for g in windows(3):
    assert {grid[i][j] for (i, j) in g.cells} == set(range(1, 10))
print("windows ok")

+---------+---------+---------+
|6  2  7 |3  1  5 |4  8  9 |
|3  1  9 |7  8  4 |2  6  5 |
|4  5  8 |2  6  9 |3  7  1 |
+---------+---------+---------+
|2  4  3 |6  9  1*|8  5  7 |
|5  9  1 |8  2  7 |6  3  4 |
|8* 7  6 |5  4  3 |9  1  2 |
+---------+---------+---------+
|1  3  2 |9  5  8 |7  4  6 |
|9  8  4 |1  7  6 |5  2  3 |
|7  6  5 |4  3  2 |1  9* 8 |
+---------+---------+---------+
windows ok


### Irregular Sudoku (jigsaw)

We replace the $n\times n$ blocks by irregular regions of the same size, each a `box` built cell by cell. Example $4\times4$ ($n=2$), with regions `A`, `B`, `C`, `D` of 4 cells each:

```
A A A B
C A B B
C C D B
C D D D
```

In [15]:
def jigsaw_regions(layout, n):
    regs = {}
    for i, line in enumerate(layout):
        for j, r in enumerate(line.split()):
            regs.setdefault(r, box(n)).add(i, j)
    assert all(len(g) == n * n for g in regs.values())
    return list(regs.values())

layout = ["A A A B",
          "C A B B",
          "C C D B",
          "C D D D"]
n = 2
N = n * n
regs = jigsaw_regions(layout, n)

# jigsaw = rows + columns + irregular regions (instead of the cubes)
m = sudoku_model(n)
m.add_groups(*[path(n, (i, 0), (i, N - 1)) for i in range(N)],
             *[path(n, (0, j), (N - 1, j)) for j in range(N)],
             *regs)
clues = box(n, {(0, 0): 1})
m.add_clues(clues)
g = m.solve()
print(m.status)
for row in g:
    print(row)
for r in regs:
    assert {g[i][j] for (i, j) in r.cells} == set(range(1, N + 1))

OPTIMAL
[1, 3, 2, 4]
[2, 4, 3, 1]
[3, 1, 4, 2]
[4, 2, 1, 3]


### Scale

The code has nothing hard-wired to $9\times9$, so it runs unchanged for $n=4$ ($16\times16$), $n=5$ ($25\times25$) and $n=6$ ($36\times36$). We measure the solving time with $k=n$ clues.

In [16]:
for n in (3, 4, 5, 6):
    t0 = time.time()
    clues, grid, att = generate_and_solve(n, seed=1, time_limit=60)
    dt = time.time() - t0
    if grid is None:
        print(f"n={n}: no result after {dt:.1f}s")
    else:
        validate(grid, n, clues)
        print(f"n={n} ({n*n}x{n*n}): {dt:.2f}s, {att} attempt(s), validated")

n=3 (9x9): 0.05s, 1 attempt(s), validated


n=4 (16x16): 0.38s, 1 attempt(s), validated


n=5 (25x25): 2.29s, 1 attempt(s), validated


n=6 (36x36): 10.77s, 1 attempt(s), validated


**Discussion of the limits.** The number of variables grows with $N^2=n^4$ and the number of *alldifferent* constraints with $3N=3n^2$, each over $N$ variables. In our measurements (with $k=n$ clues) the time went from about 0.05 s for $n=3$ to about 11 s for $n=6$, so it grows fast but remains feasible. Also, random clue generation stops being efficient when $k$ is large, as we saw in exercise 7. With few clues the grid is barely constrained and the solver finds a solution quickly because there are many; the hard case is heavily constrained puzzles with few solutions, where the propagation of `AddAllDifferent` and the search weigh more.